# Auditoria de Planos de Conta por Credor

Verifica se cada credor (`nome`) está lançando movimentações no **Plano de Conta padrão** que ele historicamente utiliza na base financeira (`base.csv`).

**Metodologia:**
- O padrão (`descdc_padrao`) de cada credor é definido pelo `descdc` mais frequente em **todo o histórico** disponível (moda global).
- Registros com data placeholder `1800-01-01` são excluídos antes da análise (legado/migração).
- Lançamentos com `descdc` diferente do padrão são sinalizados com a categoria da divergência.

**Categorias de divergência:**

| Categoria | Descrição |
|---|---|
| `DIVERGENCIA_DE_PADRAO_DO_CREDOR` | O `descdc` do lançamento difere do padrão histórico predominante do credor |
| `PADRAO_AMBIGUO` | O credor tem dois ou mais `descdc` com a mesma frequência máxima (empate de moda) |
| `BAIXA_CONFIANCA_PADRAO` | Credor com poucos lançamentos no histórico — padrão não é estatisticamente confiável |

---

## Como usar

1. Ajuste os parâmetros na **Célula 1** se necessário (caminho do arquivo, limiares).
2. Execute todas as células em sequência (`Run All`).
3. Consulte o **Relatório Resumido** (por credor) e o **Relatório Detalhado** (por lançamento).
4. A nota `auditoria_padrao_credor_YYYYMMDD.md` será exportada em `00-Zettlelkasten/` no formato Zettelkasten.

In [8]:
# ─── Célula 1 — Parâmetros ───────────────────────────────────────────────────
import os
import pandas as pd
import numpy as np
from datetime import datetime
from IPython.display import display, HTML

# Detecção automática do workspace (funciona da raiz, de 04-Notebooks/ e de subpastas)
_cwd = os.getcwd()
if os.path.isdir(os.path.join(_cwd, '02-Referencias')):
    WORKSPACE = _cwd
elif os.path.isdir(os.path.join(_cwd, '..', '02-Referencias')):
    WORKSPACE = os.path.normpath(os.path.join(_cwd, '..'))
elif os.path.isdir(os.path.join(_cwd, '..', '..', '02-Referencias')):
    WORKSPACE = os.path.normpath(os.path.join(_cwd, '..', '..'))
else:
    raise FileNotFoundError(f'Pasta 02-Referencias nao encontrada a partir de: {_cwd}')

REFS      = os.path.join(WORKSPACE, '02-Referencias')
ZETTEL    = os.path.join(WORKSPACE, '00-Zettlelkasten')

# Caminho da base principal (ODBC da empresa)
PATH_BASE = os.path.join(REFS, 'Meus Dados', 'base.csv')

# Limiar mínimo de lançamentos para o padrão ser considerado confiável.
# Credores com menos lançamentos recebem a categoria BAIXA_CONFIANCA_PADRAO.
MIN_LANCAMENTOS_CREDOR = 5

# ---------------- Parâmetros de período da auditoria ----------------
# Campo de data a usar no filtro: 'lancamento', 'ite_pagrec_vencimento' ou 'iterea_pagamento'
COLUNA_DATA_AUDITORIA = 'lancamento'

# Modo de período:
# - 'completo'  : não aplica filtro de data
# - 'ano'       : filtra um ano inteiro (ex.: 2026)
# - 'intervalo' : filtra data inicial e final (ex.: jan-mar/2026)
MODO_PERIODO = 'ano'

# Usado quando MODO_PERIODO == 'ano'
ANO_AUDITORIA = 2026

# Usado quando MODO_PERIODO == 'intervalo' (formato YYYY-MM-DD)
DATA_INICIO = '2026-01-01'
DATA_FIM    = '2026-04-02'

# Excluir intercompanies da auditoria de credores?
# Regra solicitada: desconsiderar todos os credores com G3S, G&S e RSE no nome.
EXCLUIR_INTERCOMPANIES = True
PADROES_INTERCOMPANY = ['G3S', 'G&S', 'RSE']

# Exportar nota Markdown no formato Zettelkasten ao final?
EXPORTAR_MD = True

hoje = datetime.now()

# Nome do arquivo inclui a data para permitir histórico de auditorias
PATH_EXPORT = os.path.join(ZETTEL, f'auditoria_padrao_credor_{hoje.strftime("%Y%m%d")}.md')

print(f'Workspace   : {WORKSPACE}')
print(f'Base        : {PATH_BASE}')
print(f'Min. lanç.  : {MIN_LANCAMENTOS_CREDOR}')
print(f'Data filtro : {COLUNA_DATA_AUDITORIA}')
print(f'Modo período: {MODO_PERIODO}')
if MODO_PERIODO == 'ano':
    print(f'Ano         : {ANO_AUDITORIA}')
elif MODO_PERIODO == 'intervalo':
    print(f'Intervalo   : {DATA_INICIO} até {DATA_FIM}')
print(f'Excluir IC  : {EXCLUIR_INTERCOMPANIES} ({", ".join(PADROES_INTERCOMPANY)})')
print(f'Export MD   : {EXPORTAR_MD}')
print(f'Saída MD    : {PATH_EXPORT}')
print(f'Executado em: {hoje.strftime("%d/%m/%Y %H:%M")}')

Workspace   : c:\Users\julio.santana\Documents\Projects\Cofre_Trabalho
Base        : c:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\base.csv
Min. lanç.  : 5
Data filtro : lancamento
Modo período: ano
Ano         : 2026
Excluir IC  : True (G3S, G&S, RSE)
Export MD   : True
Saída MD    : c:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\00-Zettlelkasten\auditoria_padrao_credor_20260406.md
Executado em: 06/04/2026 14:42


In [9]:
# ─── Célula 2 — Carga, limpeza e filtro de período ───────────────────────────

print('Carregando base.csv ...')
df_base = pd.read_csv(PATH_BASE, sep=';', encoding='latin-1', dtype=str)
print(f'  Registros brutos          : {len(df_base):,}')

# ── Filtro: excluir registros com data placeholder 1800-01-01 ─────────────────
# Esses registros são saldos migrados de sistema legado e não representam
# movimentações reais — incluí-los distorceria o padrão do credor.
_mask_legado = (
    df_base['ite_pagrec_vencimento'].str.strip().eq('1800-01-01') |
    df_base['iterea_pagamento'].str.strip().eq('1800-01-01')
)
n_excluidos = _mask_legado.sum()
df = df_base[~_mask_legado].copy()
print(f'  Excluídos (1800-01-01)    : {n_excluidos:,}')

# ── Filtro de período configurável ─────────────────────────────────────────────
# Converte a coluna de data escolhida para datetime; valores inválidos viram NaT.
# 'lancamento' vem em DD/MM/YYYY; as demais datas vêm em YYYY-MM-DD.
if COLUNA_DATA_AUDITORIA == 'lancamento':
    df['_data_auditoria'] = pd.to_datetime(
        df[COLUNA_DATA_AUDITORIA],
        format='%d/%m/%Y',
        errors='coerce'
    )
else:
    df['_data_auditoria'] = pd.to_datetime(df[COLUNA_DATA_AUDITORIA], errors='coerce')
antes_periodo = len(df)

if MODO_PERIODO == 'completo':
    pass
elif MODO_PERIODO == 'ano':
    inicio_ano = pd.Timestamp(f'{ANO_AUDITORIA}-01-01')
    fim_ano    = pd.Timestamp(f'{ANO_AUDITORIA}-12-31')
    df = df[df['_data_auditoria'].between(inicio_ano, fim_ano)].copy()
elif MODO_PERIODO == 'intervalo':
    dt_ini = pd.to_datetime(DATA_INICIO, errors='coerce')
    dt_fim = pd.to_datetime(DATA_FIM, errors='coerce')
    if pd.isna(dt_ini) or pd.isna(dt_fim):
        raise ValueError('DATA_INICIO e DATA_FIM devem estar no formato YYYY-MM-DD')
    if dt_ini > dt_fim:
        raise ValueError('DATA_INICIO não pode ser maior que DATA_FIM')
    df = df[df['_data_auditoria'].between(dt_ini, dt_fim)].copy()
else:
    raise ValueError("MODO_PERIODO inválido. Use: 'completo', 'ano' ou 'intervalo'.")

depois_periodo = len(df)
print(f'  Registros após período     : {depois_periodo:,}  (de {antes_periodo:,})')

# ── Normalização textual ───────────────────────────────────────────────────────
# Remove variações de espaço/caixa que causariam falsos positivos
def _norm(s):
    return s.astype(str).str.strip().str.upper().str.replace(r'\s+', ' ', regex=True)

df['nome_norm']   = _norm(df['nome'])
df['descdc_norm'] = _norm(df['descdc'])
df['codcdc_norm'] = df['codcdc'].astype(str).str.strip()

# ── Exclusão de intercompanies (G3S, G&S, RSE) ───────────────────────────────
# Aplica após normalização do nome para ficar robusto a variações de caixa/espaço.
if EXCLUIR_INTERCOMPANIES:
    padrao_ic = '|'.join(PADROES_INTERCOMPANY)
    mask_ic = df['nome_norm'].str.contains(padrao_ic, regex=True, na=False)
    n_ic = int(mask_ic.sum())
    df = df[~mask_ic].copy()
    print(f'  Excluídos intercompanies   : {n_ic:,}')

# ── Conversão de valor bruto para numérico ────────────────────────────────────
df['valor_bruto_num'] = (
    df['valor_bruto'].astype(str).str.strip()
    .str.replace(',', '.', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
    .abs()
)

print(f'  Registros para análise    : {len(df):,}')
print(f'\nCredores únicos             : {df["nome_norm"].nunique():,}')
print(f'Planos de conta únicos      : {df["descdc_norm"].nunique():,}')
print('\nBase pronta para auditoria.')

Carregando base.csv ...
  Registros brutos          : 63,750
  Excluídos (1800-01-01)    : 151
  Registros após período     : 18,420  (de 63,599)
  Excluídos intercompanies   : 671
  Registros para análise    : 17,749

Credores únicos             : 2,836
Planos de conta únicos      : 129

Base pronta para auditoria.


In [10]:
# ─── Célula 3 — Padrão histórico por credor (moda global de descdc) ───────────

def _calcular_padrao(df, col_nome='nome_norm', col_conta='descdc_norm',
                     min_lancamentos=MIN_LANCAMENTOS_CREDOR):
    """
    Calcula para cada credor:
      total_lancamentos    : total de lançamentos no histórico (após filtro legado)
      descdc_padrao        : descdc mais frequente (moda global)
      codcdc_padrao        : código do plano de conta padrão
      freq_padrao          : contagem absoluta da moda
      pct_aderencia        : % de lançamentos no padrão
      qtd_descdc_distintos : quantos planos de conta distintos o credor usou
      status_padrao        : OK | PADRAO_AMBIGUO | BAIXA_CONFIANCA_PADRAO
    """
    resultados = []

    for credor, grupo in df.groupby(col_nome):
        total    = len(grupo)
        contagem = grupo[col_conta].value_counts()
        freq_max = contagem.iloc[0]
        moda_list = contagem[contagem == freq_max].index.tolist()
        empate    = len(moda_list) > 1

        # Código do plano de conta correspondente ao padrão
        padrao_descdc = moda_list[0]
        codcdc_padrao = (
            grupo.loc[grupo[col_conta] == padrao_descdc, 'codcdc_norm']
            .mode()
        )
        codcdc_padrao = codcdc_padrao.iloc[0] if len(codcdc_padrao) > 0 else ''

        if total < min_lancamentos:
            status = 'BAIXA_CONFIANCA_PADRAO'
        elif empate:
            status        = 'PADRAO_AMBIGUO'
            padrao_descdc = ' | '.join(sorted(moda_list))
            codcdc_padrao = ''
        else:
            status = 'OK'

        pct = round(freq_max / total * 100, 1) if not empate else None

        resultados.append({
            'nome_norm'            : credor,
            'total_lancamentos'    : total,
            'descdc_padrao'        : padrao_descdc,
            'codcdc_padrao'        : codcdc_padrao,
            'freq_padrao'          : freq_max,
            'pct_aderencia'        : pct,
            'qtd_descdc_distintos' : contagem.nunique(),
            'status_padrao'        : status,
        })

    return (
        pd.DataFrame(resultados)
        .sort_values('total_lancamentos', ascending=False)
        .reset_index(drop=True)
    )


df_padrao = _calcular_padrao(df)

n_ok      = (df_padrao['status_padrao'] == 'OK').sum()
n_ambiguo = (df_padrao['status_padrao'] == 'PADRAO_AMBIGUO').sum()
n_baixa   = (df_padrao['status_padrao'] == 'BAIXA_CONFIANCA_PADRAO').sum()

print('=== Resumo dos padrões detectados ===')
print(f'  OK (padrão claro)          : {n_ok:,}')
print(f'  Padrão ambíguo (empate)    : {n_ambiguo:,}')
print(f'  Baixa confiança (< {MIN_LANCAMENTOS_CREDOR} lanç.) : {n_baixa:,}')
print(f'  Total de credores          : {len(df_padrao):,}')

print('\nTop 10 credores por volume de lançamentos:')
display(
    df_padrao[['nome_norm','total_lancamentos','descdc_padrao',
               'pct_aderencia','qtd_descdc_distintos','status_padrao']]
    .head(10)
    .style.set_table_styles([{'selector':'th','props':'background:#1f3864;color:#fff;padding:6px 10px'}])
)

=== Resumo dos padrões detectados ===
  OK (padrão claro)          : 765
  Padrão ambíguo (empate)    : 24
  Baixa confiança (< 5 lanç.) : 2,047
  Total de credores          : 2,836

Top 10 credores por volume de lançamentos:


,nome_norm,total_lancamentos,descdc_padrao,pct_aderencia,qtd_descdc_distintos,status_padrao
0,GSL LOGISTICA E TRANSPORTE LTDA,373,TRANSPORTE DE SUCATA,95.400000,4,OK
1,ROSSE E ROSSE LTDA,367,COMPRAS DE SUCATAS,100.000000,1,OK
2,BANCO BRADESCO,344,RENDIMENTO FINANCEIRO,76.200000,6,OK
3,BRADESCO AUTO / SEGURO,271,SEGUROS,100.000000,1,OK
4,BANCO COOPERATIVO SICREDI SA,243,DESPESAS BANCARIAS,62.100000,5,OK
5,GERDAU ACOS LONGOS SA,240,VENDAS DE SUCATAS,97.100000,3,OK
6,ZHEJIANG MINXIN MATERIALES DE CONSTRUCCION S.R.L SUCURSAL B,217,VENDAS DE SUCATAS,100.000000,1,OK
7,REDEPLAST RECICLAGEM LTDA,188,PESAGENS AVULSAS,98.900000,2,OK
8,CLIENTE BALCAO PR,161,VENDAS DE SUCATAS,100.000000,1,OK
9,CLIENTE BALCAO MS,148,VENDAS DE SUCATAS,100.000000,1,OK


In [11]:
# ─── Célula 4 — Detecção de divergências e classificação ─────────────────────

# Juntar cada lançamento com o padrão do seu credor
df_merged = df.merge(
    df_padrao[['nome_norm','descdc_padrao','codcdc_padrao','status_padrao','total_lancamentos','pct_aderencia']],
    on='nome_norm',
    how='left'
)

# Identificar divergências:
# Um lançamento diverge quando seu descdc_norm ≠ descdc_padrao do credor.
# Para credores com PADRAO_AMBIGUO ou BAIXA_CONFIANCA_PADRAO, a categoria
# reflete o status do padrão, não necessariamente um erro operacional.

def _classificar(row):
    status = row['status_padrao']
    if status == 'BAIXA_CONFIANCA_PADRAO':
        # Mesmo com pouco histórico, se o descdc difere do mais frequente, sinalizamos
        if row['descdc_norm'] != row['descdc_padrao']:
            return 'BAIXA_CONFIANCA_PADRAO'
        return None
    if status == 'PADRAO_AMBIGUO':
        # Qualquer lançamento deste credor é ambíguo — não há padrão único
        return 'PADRAO_AMBIGUO'
    # Status OK: verifica divergência real
    if row['descdc_norm'] != row['descdc_padrao']:
        return 'DIVERGENCIA_DE_PADRAO_DO_CREDOR'
    return None

df_merged['motivo_divergencia'] = df_merged.apply(_classificar, axis=1)

# Separar apenas os lançamentos com alguma divergência
df_div = df_merged[df_merged['motivo_divergencia'].notna()].copy()

# Manter apenas as divergências reais para o relatório principal
df_div_real = df_div[df_div['motivo_divergencia'] == 'DIVERGENCIA_DE_PADRAO_DO_CREDOR'].copy()

n_total_div  = len(df_div)
n_div_real   = len(df_div_real)
n_div_ambig  = (df_div['motivo_divergencia'] == 'PADRAO_AMBIGUO').sum()
n_div_baixa  = (df_div['motivo_divergencia'] == 'BAIXA_CONFIANCA_PADRAO').sum()
n_cred_div   = df_div_real['nome_norm'].nunique()

print('=== Resultado da detecção ===')
print(f'  Lançamentos analisados           : {len(df):,}')
print(f'  Divergências reais encontradas   : {n_div_real:,}  (em {n_cred_div:,} credores distintos)')
print(f'  Lançamentos em credor ambíguo    : {n_div_ambig:,}')
print(f'  Lançamentos baixa confiança      : {n_div_baixa:,}')
print(f'  Total sinalizado                 : {n_total_div:,}')

=== Resultado da detecção ===
  Lançamentos analisados           : 17,749
  Divergências reais encontradas   : 1,105  (em 158 credores distintos)
  Lançamentos em credor ambíguo    : 420
  Lançamentos baixa confiança      : 122
  Total sinalizado                 : 1,647


In [12]:
# ─── Célula 5 — Relatório Resumido por Credor ─────────────────────────────────

# Agrupa as divergências reais por credor, mostrando quais descdc divergentes
# foram usados, com quantos lançamentos e qual o impacto financeiro total.

resumo_rows = []

for credor, grupo in df_div_real.groupby('nome_norm'):
    info_padrao  = df_padrao[df_padrao['nome_norm'] == credor].iloc[0]
    total_hist   = int(info_padrao['total_lancamentos'])
    descdc_pad   = info_padrao['descdc_padrao']
    pct_ader     = info_padrao['pct_aderencia']

    # Quais descdc divergentes foram usados
    divergentes = (
        grupo.groupby('descdc_norm')
             .agg(qtd_lanc=('descdc_norm','count'), impacto_rs=('valor_bruto_num','sum'))
             .sort_values('qtd_lanc', ascending=False)
    )

    for descdc_div, row_div in divergentes.iterrows():
        resumo_rows.append({
            'credor'              : credor,
            'total_historico'     : total_hist,
            'descdc_padrao'       : descdc_pad,
            'pct_aderencia_padrao': pct_ader,
            'descdc_divergente'   : descdc_div,
            'qtd_lancamentos_div' : int(row_div['qtd_lanc']),
            'impacto_rs'          : round(row_div['impacto_rs'], 2),
        })

df_resumo = (
    pd.DataFrame(resumo_rows)
    .sort_values(['impacto_rs','qtd_lancamentos_div'], ascending=False)
    .reset_index(drop=True)
)

# ── Formatação HTML para exibição ─────────────────────────────────────────────
_TH  = 'background:#1f3864;color:#fff;padding:8px 12px;font-size:12px;white-space:nowrap'
_TD  = 'padding:6px 10px;border:1px solid #dee2e6;font-size:12px'
_TDR = _TD + ';text-align:right'

def _fmt_rs(v):
    return f'R$ {v:,.2f}'

def _fmt_pct(v):
    if pd.isna(v): return '—'
    return f'{v:.1f}%'

print(f'=== Relatório Resumido — {len(df_resumo)} combinações credor × descdc divergente ===')
print(f'    Credores com divergências reais: {df_resumo["credor"].nunique():,}\n')

# Exibir top 50 por impacto financeiro
df_resumo_display = df_resumo.head(50).copy()
df_resumo_display['impacto_rs']           = df_resumo_display['impacto_rs'].apply(_fmt_rs)
df_resumo_display['pct_aderencia_padrao'] = df_resumo_display['pct_aderencia_padrao'].apply(_fmt_pct)

df_resumo_display.columns = [
    'Credor', 'Total hist.', 'Plano Padrão (esperado)',
    '% Aderência', 'Plano Divergente (lançado)', 'Qtd. Lanç. Div.', 'Impacto R$'
]

styled_resumo = (
    df_resumo_display.style
    .set_table_styles([
        {'selector': 'table', 'props': 'border-collapse:collapse;width:100%;font-family:system-ui,sans-serif'},
        {'selector': 'th',    'props': _TH},
        {'selector': 'td',    'props': _TD},
        {'selector': 'td:nth-child(3)', 'props': _TD + ';color:#198754;font-weight:600'},
        {'selector': 'td:nth-child(5)', 'props': _TD + ';color:#dc3545;font-weight:600'},
        {'selector': 'td:nth-child(6)', 'props': _TDR},
        {'selector': 'td:nth-child(7)', 'props': _TDR + ';font-weight:bold'},
        {'selector': 'tr:nth-child(even)', 'props': 'background:#f8f9fa'},
        {'selector': 'tr:hover td', 'props': 'filter:brightness(0.93)'},
    ])
)

display(styled_resumo)

=== Relatório Resumido — 338 combinações credor × descdc divergente ===
    Credores com divergências reais: 158



,Credor,Total hist.,Plano Padrão (esperado),% Aderência,Plano Divergente (lançado),Qtd. Lanç. Div.,Impacto R$
0,BANCO COOPERATIVO SICREDI SA,243,DESPESAS BANCARIAS,62.1%,JUROS E MULTAS,11,"R$ 797,249.04"
1,ITATIAIA DISTRIBUIDORA DE VEICULOS LTDA,5,VEÍCULOS,80.0%,VENDA DE VEÍCULOS,1,"R$ 760,000.00"
2,ARCELOR MITTAL BRASIL S.A,37,VENDAS DE SUCATAS,83.8%,LOCACAO DE MAQUINAS E EQUIPAMENTOS,5,"R$ 685,833.16"
3,GERDAU ACOS LONGOS SA,240,VENDAS DE SUCATAS,97.1%,LOCACAO DE MAQUINAS E EQUIPAMENTOS,4,"R$ 304,000.00"
4,FFC GESTAO LTDA,37,DESPESAS DE VIAGEM,43.2%,HONORÁRIOS PJ,13,"R$ 274,486.21"
5,RENATO JOSE APARECIDO SALTEIRO,25,PRO LABORE,32.0%,JUROS E MULTAS,2,"R$ 182,660.00"
6,RENATO JOSE APARECIDO SALTEIRO,25,PRO LABORE,32.0%,EMPRÉSTIMOS,2,"R$ 182,660.00"
7,RENATO JOSE APARECIDO SALTEIRO,25,PRO LABORE,32.0%,EMPRESTIMOS,2,"R$ 180,000.00"
8,EVALDO JANES DE CANDIDO OLIVIERA LTDA,25,DESPESAS DE VIAGEM,48.0%,HONORÁRIOS PJ,5,"R$ 150,000.00"
9,BANCO COOPERATIVO SICREDI SA,243,DESPESAS BANCARIAS,62.1%,IOF,11,"R$ 146,751.87"


In [13]:
# ─── Célula 6 — Caso exemplo: MULT SEG SISTEMA ELETRONICO DE SEGURANAA ────────
# Demonstra o resultado esperado conforme o caso real identificado na análise.

CREDOR_EXEMPLO = 'MULT SEG SISTEMA ELETRONICO DE SEGURANAA'

# Busca flexível: procura substring do nome (acomoda truncamentos)
hits = df['nome_norm'][df['nome_norm'].str.contains('MULT SEG', na=False)].unique()
if len(hits) > 0:
    CREDOR_EXEMPLO = hits[0]

print(f'Credor analisado: "{CREDOR_EXEMPLO}"\n')

# Histórico completo do credor
hist = df[df['nome_norm'] == CREDOR_EXEMPLO].copy()
contagem_hist = (
    hist.groupby('descdc_norm')
        .agg(qtd=('descdc_norm','count'), valor_total=('valor_bruto_num','sum'))
        .sort_values('qtd', ascending=False)
        .reset_index()
)
contagem_hist.columns = ['Plano de Conta (descdc)', 'Qtd. Lançamentos', 'Valor Total R$']
contagem_hist['Valor Total R$'] = contagem_hist['Valor Total R$'].apply(lambda v: f'R$ {v:,.2f}')

print('Distribuição de planos de conta no histórico:')
display(contagem_hist)

# Lançamentos divergentes deste credor
div_ex = df_div_real[df_div_real['nome_norm'] == CREDOR_EXEMPLO]
if len(div_ex) > 0:
    colunas_exibir = ['lancamento','documento','filial','descdc_padrao','descdc_norm','valor_bruto_num','motivo_divergencia']
    colunas_labels = ['Data Lanç.','Documento','Filial','Plano Padrão','Plano Lançado','Valor R$','Motivo']
    div_ex_display = div_ex[colunas_exibir].copy()
    div_ex_display.columns = colunas_labels
    div_ex_display['Valor R$'] = div_ex_display['Valor R$'].apply(lambda v: f'R$ {v:,.2f}')
    print(f'\nLançamentos fora do padrão ({len(div_ex_display)}):')
    display(div_ex_display.reset_index(drop=True))
else:
    print('Nenhuma divergência encontrada para este credor no período analisado.')

Credor analisado: "MULT SEG SISTEMA ELETRONICO DE SEGURANÃA"

Distribuição de planos de conta no histórico:


,Plano de Conta (descdc),Qtd. Lançamentos,Valor Total R$
0,ALARME E MONITORAMENTO,11,"R$ 2,707.00"
1,LOCAÇÃO DE EQUIPAMENTOS E FERRAMENTAS,1,R$ 128.00



Lançamentos fora do padrão (1):


,Data Lanç.,Documento,Filial,Plano Padrão,Plano Lançado,Valor R$,Motivo
0,16/01/2026,27822,G&S MARINGA,ALARME E MONITORAMENTO,LOCAÇÃO DE EQUIPAMENTOS E FERRAMENTAS,R$ 128.00,DIVERGENCIA_DE_PADRAO_DO_CREDOR


In [14]:
# ─── Célula 7 — Relatório Detalhado por Lançamento ───────────────────────────
# Lista todas as linhas divergentes com contexto completo para auditoria operacional.

COLUNAS_DETALHE = [
    'lancamento', 'filial', 'nome_norm', 'codcdc_norm', 'descdc_norm',
    'descdc_padrao', 'codcdc_padrao', 'motivo_divergencia',
    'valor_bruto_num', 'documento', 'codcen', 'descen',
]

LABELS_DETALHE = [
    'Data Lanç.', 'Filial', 'Credor', 'Cód. PC Lançado', 'PC Lançado',
    'PC Padrão', 'Cód. PC Padrão', 'Motivo Divergência',
    'Valor R$', 'Documento', 'Cód. CC', 'Descrição CC',
]

# Evita NameError quando a célula é executada isoladamente.
if 'df_div_real' not in globals():
    if 'df_div' in globals() and 'motivo_divergencia' in df_div.columns:
        df_div_real = df_div[df_div['motivo_divergencia'] == 'DIVERGENCIA_DE_PADRAO_DO_CREDOR'].copy()
        print('Aviso: df_div_real foi recriado a partir de df_div.')
    else:
        raise RuntimeError(
            "'df_div_real' não está disponível. Execute as células anteriores até a etapa de detecção de divergências."
        )

df_detalhe = df_div_real[COLUNAS_DETALHE].copy()
df_detalhe = df_detalhe.sort_values(['nome_norm','lancamento']).reset_index(drop=True)
df_detalhe.columns = LABELS_DETALHE
df_detalhe['Valor R$'] = df_detalhe['Valor R$'].apply(lambda v: f'R$ {v:,.2f}' if pd.notna(v) else '—')

_TH2  = 'background:#1f3864;color:#fff;padding:7px 10px;font-size:11px;white-space:nowrap'
_TD2  = 'padding:5px 9px;border:1px solid #dee2e6;font-size:11px'
_TDW  = _TD2 + ';color:#dc3545;font-weight:600'
_TDOK = _TD2 + ';color:#198754;font-weight:600'

styled_det = (
    df_detalhe.head(200).style   # limita exibição inline a 200 linhas
    .set_table_styles([
        {'selector': 'table',             'props': 'border-collapse:collapse;width:100%;font-family:system-ui,sans-serif'},
        {'selector': 'th',                'props': _TH2},
        {'selector': 'td',                'props': _TD2},
        {'selector': 'td:nth-child(5)',   'props': _TDW},
        {'selector': 'td:nth-child(6)',   'props': _TDOK},
        {'selector': 'tr:nth-child(even)','props': 'background:#f8f9fa'},
        {'selector': 'tr:hover td',       'props': 'filter:brightness(0.92)'},
    ])
)

print(f'=== Relatório Detalhado — {len(df_div_real):,} lançamentos divergentes ===')
if len(df_div_real) > 200:
    print(f'    (exibindo primeiros 200 — use o CSV exportado para o conjunto completo)')
print()
display(styled_det)

=== Relatório Detalhado — 1,105 lançamentos divergentes ===
    (exibindo primeiros 200 — use o CSV exportado para o conjunto completo)



,Data Lanç.,Filial,Credor,Cód. PC Lançado,PC Lançado,PC Padrão,Cód. PC Padrão,Motivo Divergência,Valor R$,Documento,Cód. CC,Descrição CC
0,04/02/2026,G3S PRUDENTE,59.179.222 BRYAN NASCIMENTO DOS SANTOS,6.1.1,COMPRAS DE SUCATAS,VENDAS DE SUCATAS,4.1.1,DIVERGENCIA_DE_PADRAO_DO_CREDOR,R$ 231.00,BOLC-292428,1.2.5.2,GERAL CONSOLIDADO / SELETIVA / PRESIDENTE PRUDENTE / COMERCIAL
1,03/02/2026,G3S DOURADOS,ADEILDO GALINDO DA SILVA,6.1.1,COMPRAS DE SUCATAS,VENDAS DE SUCATAS,4.1.1,DIVERGENCIA_DE_PADRAO_DO_CREDOR,R$ 962.00,BOLC-292275,1.2.2.2,GERAL CONSOLIDADO / SELETIVA / DOURADOS / COMERCIAL
2,05/02/2026,G3S DOURADOS,ADEILDO GALINDO DA SILVA,6.1.1,COMPRAS DE SUCATAS,VENDAS DE SUCATAS,4.1.1,DIVERGENCIA_DE_PADRAO_DO_CREDOR,"R$ 1,328.00",BOLC-292478,1.2.2.2,GERAL CONSOLIDADO / SELETIVA / DOURADOS / COMERCIAL
3,09/02/2026,G3S DOURADOS,ADEILDO GALINDO DA SILVA,6.1.1,COMPRAS DE SUCATAS,VENDAS DE SUCATAS,4.1.1,DIVERGENCIA_DE_PADRAO_DO_CREDOR,R$ 588.00,BOLC-292783,1.2.2.2,GERAL CONSOLIDADO / SELETIVA / DOURADOS / COMERCIAL
4,11/03/2026,G3S DOURADOS,ADEILDO GALINDO DA SILVA,6.1.1,COMPRAS DE SUCATAS,VENDAS DE SUCATAS,4.1.1,DIVERGENCIA_DE_PADRAO_DO_CREDOR,"R$ 1,168.00",BOLC-295385,1.2.2.2,GERAL CONSOLIDADO / SELETIVA / DOURADOS / COMERCIAL
5,12/01/2026,G3S DOURADOS,ADEILDO GALINDO DA SILVA,6.1.1,COMPRAS DE SUCATAS,VENDAS DE SUCATAS,4.1.1,DIVERGENCIA_DE_PADRAO_DO_CREDOR,R$ 322.00,BOLC-290326,1.2.2.2,GERAL CONSOLIDADO / SELETIVA / DOURADOS / COMERCIAL
6,13/02/2026,G3S DOURADOS,ADEILDO GALINDO DA SILVA,6.1.1,COMPRAS DE SUCATAS,VENDAS DE SUCATAS,4.1.1,DIVERGENCIA_DE_PADRAO_DO_CREDOR,"R$ 1,584.00",BOLC-293273,1.2.2.2,GERAL CONSOLIDADO / SELETIVA / DOURADOS / COMERCIAL
7,15/01/2026,G3S DOURADOS,ADEILDO GALINDO DA SILVA,6.1.1,COMPRAS DE SUCATAS,VENDAS DE SUCATAS,4.1.1,DIVERGENCIA_DE_PADRAO_DO_CREDOR,R$ 672.00,BOLC-290617,1.2.2.2,GERAL CONSOLIDADO / SELETIVA / DOURADOS / COMERCIAL
8,17/03/2026,G3S DOURADOS,ADEILDO GALINDO DA SILVA,6.1.1,COMPRAS DE SUCATAS,VENDAS DE SUCATAS,4.1.1,DIVERGENCIA_DE_PADRAO_DO_CREDOR,R$ 896.00,BOLC-295895,1.2.2.2,GERAL CONSOLIDADO / SELETIVA / DOURADOS / COMERCIAL
9,17/03/2026,G3S DOURADOS,ADEILDO GALINDO DA SILVA,6.1.1,COMPRAS DE SUCATAS,VENDAS DE SUCATAS,4.1.1,DIVERGENCIA_DE_PADRAO_DO_CREDOR,R$ 160.00,BOLC-295914,1.2.2.2,GERAL CONSOLIDADO / SELETIVA / DOURADOS / COMERCIAL


In [15]:
# ─── Célula 8 — Export Markdown (Zettelkasten) e sumário final ──────────────

impacto_total = df_div_real['valor_bruto_num'].sum()
top_impacto = (
    df_div_real.groupby('nome_norm')['valor_bruto_num']
               .sum()
               .sort_values(ascending=False)
)
top3 = top_impacto.head(3)

# ── Geração da nota Markdown no formato Zettelkasten ──────────────────────────
if EXPORTAR_MD:
    linhas_md = []

    # Frontmatter
    linhas_md += [
        '---',
        'tags:',
        '  - note',
        '  - controladoria',
        '  - auditoria',
        '  - plano-de-contas',
        '  - credores',
        '---',
        f'{hoje.strftime("%d/%m/%y")} - {hoje.strftime("%H:%M")}',
        '',
        '',
        '___',
        '',
    ]

    # Título e contextualização
    linhas_md += [
        f'# ~={{Titulo}}Auditoria de Planos de Conta por Credor=~',
        '',
        f'> Relatório gerado automaticamente pelo notebook `04-Notebooks/auditoria_padrao_credor.ipynb`  ',
        f'> **Data:** {hoje.strftime("%d/%m/%Y %H:%M")} | **Base:** `02-Referencias/Meus Dados/base.csv`',
        '',
        '---',
        '',
    ]

    # Resumo executivo
    linhas_md += [
        '### ~={Titulo}Resumo Executivo=~',
        '',
        '| Indicador | Valor |',
        '|---|---|',
        f'| Registros analisados | =={len(df):,}== |',
        f'| Credores auditados | {df["nome_norm"].nunique():,} |',
        f'| Credores com padrão claro (OK) | {n_ok:,} |',
        f'| Credores com padrão ambíguo | {n_ambiguo:,} |',
        f'| Credores com baixo histórico (< {MIN_LANCAMENTOS_CREDOR} lanç.) | {n_baixa:,} |',
        f'| ~={{red}}Divergências reais encontradas=~ | =={n_div_real:,}== em =={n_cred_div:,}== credores |',
        f'| ~={{red}}Impacto financeiro total=~ | ==R$ {impacto_total:,.2f}== |',
        '',
        '---',
        '',
    ]

    # Top credores por impacto
    linhas_md += [
        '### ~={Titulo}Top Credores por Impacto Financeiro=~',
        '',
        '| # | Credor | Qtd. Div. | Impacto R$ |',
        '|---|---|---|---|',
    ]
    for rank, (cred, val) in enumerate(
        top_impacto.head(20).items(), start=1
    ):
        qtd = int((df_div_real['nome_norm'] == cred).sum())
        linhas_md.append(f'| {rank} | {cred} | {qtd} | R$ {val:,.2f} |')
    linhas_md += ['', '---', '']

    # Detalhe por credor (ordenado por impacto)
    linhas_md += [
        '### ~={Titulo}Divergências por Credor=~',
        '',
    ]

    credores_ordenados = top_impacto.index.tolist()

    for credor in credores_ordenados:
        info  = df_padrao[df_padrao['nome_norm'] == credor].iloc[0]
        grupo = df_div_real[df_div_real['nome_norm'] == credor].sort_values('lancamento')
        impacto_cred = grupo['valor_bruto_num'].sum()
        pct_str = f"{info['pct_aderencia']:.1f}%" if pd.notna(info['pct_aderencia']) else '—'

        linhas_md += [
            f"#### ~={{Titulo}}{credor}=~",
            '',
            f"**Plano padrão:** `{info['descdc_padrao']}` | "
            f"**Aderência:** {pct_str} | "
            f"**Histórico:** {int(info['total_lancamentos'])} lançamentos | "
            f"**Impacto divergente:** ~={{red}}R$ {impacto_cred:,.2f}=~",
            '',
            '| Data Lanç. | Documento | Filial | ~={red}Plano Lançado=~ | ~={green}Plano Padrão=~ | Valor R$ |',
            '|---|---|---|---|---|---|',
        ]

        for _, row in grupo.iterrows():
            val_str = f"R$ {row['valor_bruto_num']:,.2f}" if pd.notna(row['valor_bruto_num']) else '—'
            linhas_md.append(
                f"| {row['lancamento']} "
                f"| {row['documento']} "
                f"| {row['filial']} "
                f"| ~={{red}}{row['descdc_norm']}=~ "
                f"| ~={{green}}{row['descdc_padrao']}=~ "
                f"| {val_str} |"
            )
        linhas_md.append('')

    # Rodapé com links relacionados
    linhas_md += [
        '---',
        '',
        '___',
        '',
        '**Links relacionados:**',
        '- [[Analise Base Financeira]]',
        '- [[Tipos de Movimentação]]',
        '- [[Guia SAGI]]',
    ]

    with open(PATH_EXPORT, 'w', encoding='utf-8') as f:
        f.write('\n'.join(linhas_md))
    print(f'Nota exportada: {PATH_EXPORT}')
    print(f'  Credores com divergência detalhados: {len(credores_ordenados):,}')

# ── Sumário visual final ──────────────────────────────────────────────────────
box_style = (
    'background:linear-gradient(120deg,#f8d7da,#f5c6cb);color:#721c24;'
    'padding:16px 20px;border-radius:10px;border-left:5px solid #dc3545;'
    'font-size:1em;margin-top:10px'
)
if n_div_real == 0:
    box_style = (
        'background:linear-gradient(120deg,#d4edda,#c3e6cb);color:#155724;'
        'padding:16px 20px;border-radius:10px;border-left:5px solid #198754;'
        'font-size:1em;margin-top:10px'
    )

top3_html = ''.join(
    f'<li><b>{cred}</b> — R$ {val:,.2f}</li>'
    for cred, val in top3.items()
)

display(HTML(f'''
<div style="{box_style}">
  <b>Auditoria de Planos de Conta por Credor</b> &nbsp;|&nbsp; {hoje.strftime("%d/%m/%Y %H:%M")}<br/><br/>
  Lançamentos analisados   : <b>{len(df):,}</b><br/>
  Credores auditados       : <b>{df["nome_norm"].nunique():,}</b><br/>
  Divergências reais       : <b>{n_div_real:,}</b> em <b>{n_cred_div:,}</b> credores distintos<br/>
  Impacto financeiro total : <b>R$ {impacto_total:,.2f}</b><br/><br/>
  <b>Top 3 credores com maior impacto:</b>
  <ul style="margin:6px 0 0 0">{top3_html}</ul>
  {'<br/><i>Nota Zettelkasten exportada: ' + os.path.basename(PATH_EXPORT) + '</i>' if EXPORTAR_MD else ''}
</div>
'''))

Nota exportada: c:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\00-Zettlelkasten\auditoria_padrao_credor_20260406.md
  Credores com divergência detalhados: 158
